# 🚀 ORLITH AI — Google Colab L4 GPU Server Setup

Notebook ini mempersiapkan dan menjalankan **Backend Orlith (DocuMind AI)** secara penuh di Google Colab menggunakan akselerasi **NVIDIA L4 GPU (24GB VRAM)**.

### Komponen yang berjalan di dalam Colab:
1. **Local LLM Engine**: **Ollama** (`qwen2.5:7b` / `qwen2.5:14b` / `llama3.1:8b`) berjalan di VRAM L4.
2. **Local Embedding & Reranker**: `BAAI/bge-m3` dan `BAAI/bge-reranker-base` berjalan dengan PyTorch CUDA.
3. **SOTA OCR Engine**: **Baidu Unlimited-OCR** (3.3B parameter, R-SWA multi-page, structured markdown) berjalan di GPU L4.
4. **SOTA RAG 2.0**: Multilingual Query Expansion, HyDE, Chunk Stitching, Parent-Child Context Injection, dan Multi-List RRF.
5. **FastAPI Backend Server**: Menjalankan seluruh pipeline RAG, Hybrid Search, dan Document Processing.
6. **Cloudflare Tunnel (`cloudflared`)**: Menghasilkan URL publik HTTPS gratis untuk dihubungkan langsung ke Frontend Anda (Vercel atau localhost).

## ⚙️ Langkah 1: Cek Akselerator GPU L4
Pastikan runtime Anda sudah menggunakan **GPU L4** (*Runtime -> Change runtime type -> Hardware accelerator: GPU -> GPU type: L4*).

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name    : {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM     : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 💾 Langkah 2: Setup Storage Data Lokal Colab
Menyiapkan folder penyimpanan lokal di `/content/orlith_data` untuk database SQLite, ChromaDB vector store, dan file uploads.

In [ ]:
import os

PERSIST_DATA_DIR = "/content/orlith_data"
os.makedirs(f"{PERSIST_DATA_DIR}/chroma", exist_ok=True)
os.makedirs(f"{PERSIST_DATA_DIR}/uploads", exist_ok=True)
print(f"✅ Direktori data siap digunakan: {PERSIST_DATA_DIR}")


## 📥 Langkah 3: Ekstrak Source Code Backend (`backend.zip`)
Silakan drag-and-drop file **`backend.zip`** dari laptop Anda ke panel **Files (📁)** di sidebar sebelah kiri Colab, lalu jalankan cell ini.

In [ ]:
import os
import sys

ZIP_PATH = "/content/backend.zip"

if not os.path.exists(ZIP_PATH):
    print("👉 /content/backend.zip belum ditemukan. Membuka dialog upload file...")
    try:
        from google.colab import files
        uploaded = files.upload()
    except Exception as e:
        pass

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        "❌ File /content/backend.zip TIDAK DITEMUKAN!\n"
        "👉 Silakan drag-and-drop file backend.zip dari laptop Anda ke panel Files (📁) di sebelah kiri Colab, lalu jalankan ulang cell ini."
    )

print("📦 Mengekstrak backend.zip langsung ke /content/orlith_backend...")
!rm -rf /content/orlith_backend
!mkdir -p /content/orlith_backend
!unzip -q -o /content/backend.zip -d /content/orlith_backend

# Jika file backend.zip berisi folder nested 'backend', pindahkan ke root
if os.path.exists("/content/orlith_backend/backend/main.py"):
    !mv /content/orlith_backend/backend/* /content/orlith_backend/ 2>/dev/null || true
    !rm -rf /content/orlith_backend/backend

print("✅ Backend berhasil diekstrak ke /content/orlith_backend!")

# Pindah direktori kerja ke /content/orlith_backend
%cd /content/orlith_backend
!ls -la


## 📦 Langkah 4: Install Dependencies & Cloudflare Tunnel
Install dependensi sistem (`zstd`, `curl`, `pciutils`, `poppler-utils`), PyTorch CUDA 12.1, Pillow terbaru, Transformers 4.x, Baidu OCR dependencies, dan binary `cloudflared`.

In [ ]:
%%bash
cd /content/orlith_backend 2>/dev/null || true

echo "=== [1/6] Menginstall dependensi sistem Linux (zstd, curl, pciutils, poppler-utils) ==="
apt-get update -qq && apt-get install -y -qq zstd curl pciutils poppler-utils

echo "=== [2/6] Membersihkan cache model EasyOCR lama (jika ada) ==="
rm -rf ~/.EasyOCR/model/*.zip ~/.EasyOCR/model/*.pth 2>/dev/null || true

echo "=== [3/6] Memperbarui Pillow (mencegah error _Ink) ==="
pip install --quiet --upgrade Pillow

echo "=== [4/6] Menginstall PyTorch dengan CUDA 12.1 ==="
pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

echo "=== [5/6] Menginstall dependensi HuggingFace, Transformers 4.x, Baidu OCR & Backend ==="
pip install --quiet "huggingface-hub>=1.23.0,<2.0" "transformers>=4.48.0,<5.0.0" sentence-transformers einops addict easydict pymupdf psutil
pip install --quiet -r /content/orlith_backend/requirements.txt

echo "=== [6/6] Menyiapkan Cloudflare Tunnel (cloudflared) ==="
if ! command -v cloudflared &> /dev/null; then
    wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    rm -f cloudflared-linux-amd64.deb
fi

echo "✅ Semua dependensi, Baidu Unlimited-OCR libraries & cloudflared SIAP 100%!"


## 🦙 Langkah 5: Install & Jalankan Ollama Engine (GPU Background Daemon)
Ollama akan berjalan di latar belakang memanfaatkan GPU NVIDIA L4 pada port `11434`.

In [ ]:
%%bash
# Install Ollama CLI jika belum terinstall
if ! command -v ollama &> /dev/null; then
    curl -fsSL https://ollama.com/install.sh | sh
fi

# Jalankan daemon Ollama di background
pkill ollama || true
nohup ollama serve > /content/ollama.log 2>&1 &
echo "✅ Daemon Ollama sedang dijalankan di background..."


### Pull Model Lokal ke Ollama
Rekomendasi untuk GPU L4 (24GB VRAM):
- `qwen2.5:7b` (Sangat cepat ~4.5GB VRAM, terbaik untuk Bahasa Indonesia & JSON)
- `qwen2.5:14b` (Akurasi RAG tinggi, butuh ~9GB VRAM — L4 sangat sanggup)
- `llama3.1:8b`

In [ ]:
import time, urllib.request

# Tunggu hingga daemon Ollama siap merespons
for _ in range(20):
    try:
        urllib.request.urlopen("http://localhost:11434/")
        print("✅ Ollama daemon aktif dan siap digunakan!")
        break
    except Exception:
        time.sleep(1)

# Model yang akan diunduh ke GPU
CHOSEN_MODEL = "qwen2.5:7b"
print(f"📥 Mengunduh model {CHOSEN_MODEL} ke GPU L4...")
!ollama pull {CHOSEN_MODEL}

!ollama list


## ⚙️ Langkah 6: Konfigurasi Environment File (`.env`)
Mengatur backend untuk menggunakan:
- LLM Lokal: `qwen2.5:7b` via Ollama
- Embedding Lokal: `BAAI/bge-m3` via HuggingFace Sentence-Transformers (CUDA)
- Reranker Lokal: `BAAI/bge-reranker-base` (CUDA)
- SOTA RAG 2.0: HyDE, Chunk Stitching, Multi-list RRF, Parent-Child Context Injection
- SOTA OCR: Baidu Unlimited-OCR (CUDA bfloat16)
- Storage lokal di `/content/orlith_data`

In [ ]:
import os
import secrets

# Pastikan folder backend ada sebelum menulis .env
os.makedirs("/content/orlith_backend", exist_ok=True)

secret_key = secrets.token_hex(32)
env_content = f"""ENVIRONMENT=development
SECRET_KEY={secret_key}
DATABASE_URL=sqlite+aiosqlite:///{PERSIST_DATA_DIR}/documind.db
CHROMA_PERSIST_DIR={PERSIST_DATA_DIR}/chroma
STORAGE_BACKEND=local
STORAGE_LOCAL_PATH={PERSIST_DATA_DIR}/uploads

# AI Model Configuration (100% Local GPU)
LLM_PROVIDER=ollama
LLM_MODEL=qwen2.5:7b
OLLAMA_BASE_URL=http://localhost:11434
EMBEDDING_PROVIDER=huggingface
EMBEDDING_MODEL=BAAI/bge-m3

# SOTA RAG 2.0 Pipeline Settings
ENABLE_RERANKER=true
RERANKER_MODEL=BAAI/bge-reranker-base
ENABLE_SEMANTIC_CHUNKING=true
ENABLE_HYBRID_SEARCH=true
ENABLE_PARENT_CHILD_CHUNKING=true
ENABLE_HYDE=true
ENABLE_CHUNK_STITCHING=true
VECTOR_SEARCH_DISTANCE_CUTOFF=0.85
RAG_SIMILARITY_THRESHOLD=0.35

# SOTA OCR Engine (Baidu Unlimited-OCR on NVIDIA L4 GPU)
BAIDU_OCR_MODEL_PATH=baidu/Unlimited-OCR

# Network & CORS
CORS_ORIGINS=*
LOG_LEVEL=INFO
"""

with open("/content/orlith_backend/.env", "w", encoding="utf-8") as f:
    f.write(env_content)

print("✅ File /content/orlith_backend/.env berhasil dibuat dan dikonfigurasi!")


## 👁️ Langkah 6b: Pre-warm & Verifikasi Baidu Unlimited-OCR di GPU L4 (Opsional tapi Direkomendasikan)
Mengunduh model `baidu/Unlimited-OCR` (3.3B parameter, ~6.7 GB VRAM bfloat16) ke GPU NVIDIA L4.
Model ini akan otomatis memproses dokumen PDF & gambar menjadi **Structured Markdown** (tabel, heading, rumus utuh) yang sangat powerful untuk RAG.

In [ ]:
import typing
import torch

# 1. Patch kompatibilitas Pillow & torchvision
try:
    import PIL._typing
    if not hasattr(PIL._typing, "_Ink"):
        PIL._typing._Ink = typing.Union[int, float, tuple, str]
except Exception:
    pass

# 2. Patch kompatibilitas transformers
try:
    import transformers.utils.import_utils as t_import_utils
    if not hasattr(t_import_utils, "is_torch_fx_available"):
        t_import_utils.is_torch_fx_available = lambda: True
except Exception:
    pass

try:
    import transformers.utils as t_utils
    if not hasattr(t_utils, "is_torch_fx_available"):
        t_utils.is_torch_fx_available = lambda: True
except Exception:
    pass

from transformers import AutoModel, AutoTokenizer

print("📥 Memuat Baidu Unlimited-OCR ke GPU NVIDIA L4 (bfloat16)...")
model_name = "baidu/Unlimited-OCR"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype=torch.bfloat16,
).eval().cuda()

allocated = torch.cuda.memory_allocated(0) / (1024**3)
reserved = torch.cuda.memory_reserved(0) / (1024**3)
print(f"🎉 SUKSES! Baidu Unlimited-OCR berhasil dimuat ke GPU L4!")
print(f"📊 VRAM Terpakai: {allocated:.2f} GB / 24 GB (Reserved: {reserved:.2f} GB)")
print(f"💡 Sisa VRAM ~{24 - reserved:.1f} GB sangat longgar untuk LLM Qwen dan BGE-M3!")


## 🚀 Langkah 7: Jalankan Cloudflare Tunnel & FastAPI Backend Server
Cell ini akan:
1. Membuka tunnel publik HTTPS Cloudflare gratis mengarah ke port `8000`.
2. Menampilkan URL publik yang harus dimasukkan ke Frontend Anda.
3. Menjalankan server FastAPI backend `main:app` secara langsung.

In [ ]:
import subprocess
import time
import re
import sys
import os

# Pastikan berada di direktori backend
%cd /content/orlith_backend

# 1. Jalankan Cloudflare Tunnel di background dan tangkap URL publiknya
print("🌐 Membuka Cloudflare Tunnel ke port 8000...")
tunnel_log_path = "/content/tunnel.log"
if os.path.exists(tunnel_log_path):
    os.remove(tunnel_log_path)

pkill = subprocess.run(["pkill", "cloudflared"], capture_output=True)
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open(tunnel_log_path, "w"),
    stderr=subprocess.STDOUT
)

public_url = None
print("⏳ Menunggu URL publik dari Cloudflare (maksimal 30 detik)...")
for attempt in range(30):
    time.sleep(1)
    if os.path.exists(tunnel_log_path):
        with open(tunnel_log_path, "r", errors="ignore") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                public_url = match.group(0)
                break

if not public_url:
    print("⚠️ URL belum terbaca otomatis dari log. Silakan cek /content/tunnel.log.")
else:
    print("\n" + "="*65)
    print(f"🎉 PUBLIC BACKEND URL ANDA: {public_url}")
    print(f"📑 Interactive API Docs : {public_url}/docs")
    print(f"❤️ Health Check Endpoint : {public_url}/health")
    print("="*65)
    print(f"👉 Masukkan URL ini ke file .env Frontend Anda (localhost / Vercel):")
    print(f"   NEXT_PUBLIC_API_URL={public_url}")
    print("="*65 + "\n")

# 2. Jalankan FastAPI Server via uvicorn
print("🚀 Menjalankan FastAPI Server Orlith AI...")
!uvicorn main:app --host 0.0.0.0 --port 8000
